 <style>
.markdown-content {
    direction: rtl;
    text-align: right;
}
.markdown-content pre {
    direction: ltr;
    text-align: left;
}
.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>
<div class="markdown-content">

# Text Extraction and Cleaning در RAG

## ۱. تعریف

**Text Extraction** فرایند استخراج متن از منابعی مانند `PDF`، `TXT`، `Markdown` و `HTML` است.

**Text Cleaning** فرایند شناسایی و اصلاح مشکلات متن استخراج‌شده، پیش از ورود به مراحل بعدی پردازش در سیستم `RAG` است.

هدف این دو مرحله، آماده‌سازی متنی است که تا حد امکان محتوای اصلی سند را حفظ کند و برای `Chunking`، تولید `Embeddings` و `Retrieval` مناسب باشد.

## ۲. جایگاه در معماری RAG

جریان معمول پردازش اسناد به این شکل است:

`Document → Extraction → Cleaning → Chunking → Embeddings → Indexing`

متن استخراج‌شده مستقیماً به مدل Embedding ارسال نمی‌شود، مگر اینکه قبلاً کیفیت آن بررسی شده باشد.

## ۳. روش‌های استخراج متن

- `TXT`: خواندن محتوای فایل با مدیریت صحیح Encoding.
- `Markdown`: خواندن متن همراه با حفظ عنوان‌ها، فهرست‌ها و ساختار معنادار.
- `HTML`: استخراج محتوای اصلی و حذف عناصر غیرمرتبط با استفاده از ابزارهای مناسب.
- `Text-based PDF`: استفاده از کتابخانه‌هایی مانند `pypdf`.
- `Scanned PDF`: استفاده از `OCR` برای تشخیص متن موجود در تصاویر.

استخراج متن از PDFهای پیچیده ممکن است ترتیب متن، جدول‌ها یا ساختار صفحه را به‌درستی حفظ نکند. در چنین مواردی باید ابزار استخراج و کیفیت خروجی ارزیابی شود.

## ۴. مشکلات متداول متن خام

- `Extra Whitespace`: فاصله‌های اضافی بین کلمات یا در ابتدا و انتهای خطوط.
- `Line Breaks`: شکست خطوط نامناسب در وسط پاراگراف.
- `Unicode Issues`: وجود نمایش‌های متفاوت یا کاراکترهای نامناسب Unicode.
- `Encoding Problems`: نمایش نادرست حروف به دلیل تفسیر اشتباه Encoding.
- `Duplicate Text`: تکرار ناخواسته‌ی متن.
- `Extraction Errors`: حذف، جابه‌جایی یا اتصال اشتباه کلمات هنگام استخراج.
- `Structural Loss`: از بین رفتن ارتباط میان عنوان‌ها، جدول‌ها، فهرست‌ها و محتوای مربوط به آن‌ها.

## ۵. اصول مهم Cleaning

۱. فاصله‌های اضافی را در محدوده‌ای که ساختار متن اجازه می‌دهد اصلاح کن.

۲. شکست خطوط را با توجه به مرز پاراگراف‌ها، عنوان‌ها و فهرست‌ها مدیریت کن.

۳. در صورت نیاز، نمایش‌های Unicode را استانداردسازی کن.

۴. از حذف بی‌دلیل نشانه‌های نگارشی، اعداد، کلمات کوتاه و کاراکترهای غیر ASCII خودداری کن.

۵. معنای اصلی متن را حفظ کن؛ حذف کلماتی مانند `not` می‌تواند معنای جمله را معکوس کند.

۶. متن اصلی و متن پاک‌سازی‌شده را در صورت نیاز جداگانه نگه دار.

۷. شماره‌ی صفحه، منبع سند و سایر اطلاعات لازم برای ردیابی را حفظ کن.

۸. برای PDFهای اسکن‌شده یا دارای ساختار پیچیده، به‌جای پاک‌سازی بیشتر، کیفیت ابزار استخراج را بررسی کن.

## ۶. نمونه کد پاک‌سازی

```python
import re
import unicodedata


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", "")
    text = text.replace("\u00A0", " ")

    lines = [line.strip() for line in text.splitlines()]

    paragraphs = []
    current = []

    for line in lines:
        if line:
            current.append(line)
        elif current:
            paragraphs.append(" ".join(current))
            current = []

    if current:
        paragraphs.append(" ".join(current))

    text = "\n\n".join(paragraphs)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()
```

این تابع برای متن ساده‌ای مناسب است که شکست خطوط داخل پاراگراف‌ها باید اصلاح شود. برای جدول‌ها، کدها، فهرست‌ها و اسناد ساختاریافته، روش Cleaning باید متناسب با نوع سند طراحی شود.

## ۷. ارتباط با کیفیت RAG

کیفیت استخراج و پاک‌سازی روی مراحل بعدی تأثیر می‌گذارد:

- `Chunking`: متن خراب ممکن است مرزهای نامناسبی برای Chunkها ایجاد کند.
- `Embeddings`: مدل، متن ورودی را به بردار تبدیل می‌کند؛ اطلاعاتی که حذف شده‌اند قابل بازیابی نیستند.
- `Retrieval`: کلمات یا روابط معنایی خراب ممکن است بازیابی اطلاعات مرتبط را دشوار کنند.
- `Generation`: مدل ممکن است با اطلاعات ناقص یا اشتباه پاسخ دهد.
- `Source Attribution`: از بین رفتن اطلاعات منبع یا شماره‌ی صفحه، ردیابی پاسخ را دشوار می‌کند.

## ۸. نکته‌ی Production

در یک سیستم واقعی، مراحل `Extraction`، `Cleaning` و `Chunking` بهتر است از یکدیگر جدا باشند. خروجی هر مرحله باید قابل بررسی باشد و اطلاعاتی مانند نام فایل، شماره‌ی صفحه، وضعیت پردازش و خطاهای احتمالی حفظ شوند.

**اصل کلیدی:** Cleaning باید متن را برای پردازش آماده کند، نه اینکه اطلاعات یا معنای اصلی سند را تغییر دهد.

</div>

<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}
.markdown-content pre {
    direction: ltr;
    text-align: left;
}
.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>
<div class="markdown-content">

# تمرین نظری — Text Extraction and Cleaning

فرض کن سیستم RAG شرکت، اسناد منابع انسانی را پردازش می‌کند. متن استخراج‌شده از یک سند به شکل زیر است:

```text
Annual Leave Policy

Employees can take annual leave after
six months of employment.

Employees are NOT eligible for
unpaid leave during probation.

Contact HR for further details.
```

به پرسش‌های زیر پاسخ بده:

۱. کدام مشکلات احتمالی در متن ممکن است به `Cleaning` نیاز داشته باشند؟ کدام بخش‌ها باید بدون تغییر حفظ شوند؟

۲. اگر تمام `newline`ها و نشانه‌های نگارشی را بدون بررسی حذف کنیم، چه مشکلاتی ممکن است به وجود بیاید؟

۳. چرا باید متن اصلی را قبل از `Cleaning` نگه داریم؟

</div>

<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}
.markdown-content pre {
    direction: ltr;
    text-align: left;
}
.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>
<div class="markdown-content">

# پاسخ تمرین نظری — Text Extraction and Cleaning

## پاسخ سؤال ۱

مواردی که ممکن است به پاک‌سازی نیاز داشته باشند:

- شکست خطوط نامناسب در وسط پاراگراف‌ها.
- فاصله‌های اضافی ابتدا یا انتهای خطوط.
- فاصله‌های متوالی داخل جمله‌ها.

مواردی که باید حفظ شوند:

- عنوان `Annual Leave Policy` برای حفظ ساختار سند.
- مرز پاراگراف‌ها برای جدا نگه‌داشتن بخش‌های مختلف.
- کلمه‌ی `NOT`، چون حذف آن معنای جمله را برعکس می‌کند.
- نشانه‌های نگارشی معنادار و محتوای اصلی جملات.

## پاسخ سؤال ۲

حذف تمام شکست خطوط و نشانه‌های نگارشی بدون بررسی می‌تواند باعث چسبیدن عنوان به متن، از بین رفتن مرز پاراگراف‌ها و دشوار شدن تشخیص پایان جمله‌ها شود. در اسناد پیچیده‌تر، ممکن است ساختار فهرست‌ها و جدول‌ها نیز از بین برود.

بنابراین، پاک‌سازی باید متناسب با نوع سند انجام شود.

## پاسخ سؤال ۳

نگه‌داشتن متن اولیه (`raw_text`) این مزایا را دارد:

۱. مقایسه‌ی متن قبل و بعد از پاک‌سازی.

۲. شناسایی خطاهایی که ممکن است هنگام `Cleaning` ایجاد شده باشند.

۳. اشکال‌زدایی (`Debugging`) و تشخیص اینکه مشکل از استخراج بوده یا پاک‌سازی.

۴. امکان ردیابی متن به سند اصلی و شماره‌ی صفحه.

</div>

<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}
.markdown-content pre {
    direction: ltr;
    text-align: left;
}
.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>
<div class="markdown-content">

# تمرین عملی — Text Extraction and Cleaning

تابع `clean_text()` را با زبان Python پیاده‌سازی کن.

## ورودی

```python
raw_text = """
  Company   Leave Policy

Employees can request annual
leave after six months.

Employees are NOT eligible for
unpaid leave during probation.

Contact HR for further details.
"""
```

## الزامات

۱. فاصله‌های ابتدا و انتهای خطوط را حذف کن.

۲. فاصله‌های متوالی داخل خطوط را به یک فاصله تبدیل کن.

۳. شکست خطوط داخل هر پاراگراف را اصلاح کن.

۴. مرز پاراگراف‌ها را حفظ کن.

۵. کلمه‌ی `NOT` و معنای جمله را حفظ کن.

## کد اولیه

```python
import re


def clean_text(text: str) -> str:
    # TODO: Normalize the text
    # TODO: Process lines
    # TODO: Preserve paragraph boundaries
    # TODO: Return cleaned text
    pass


cleaned_text = clean_text(raw_text)
print(cleaned_text)
```

</div>

In [2]:
import re
import unicodedata


def clean_text(text: str) -> str:
    # Normalize Unicode
    text = unicodedata.normalize("NFC", text)

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove null characters
    text = text.replace("\x00", "")

    # Normalize non-breaking spaces
    text = text.replace("\u00A0", " ")

    # Trim each line
    lines = [line.strip() for line in text.split("\n")]

    paragraphs = []
    current_paragraph = []

    for line in lines:
        if line:
            # Collapse repeated spaces and tabs
            line = re.sub(r"[ \t]+", " ", line)
            current_paragraph.append(line)

        elif current_paragraph:
            # Preserve paragraph boundaries
            paragraphs.append(" ".join(current_paragraph))
            current_paragraph = []

    # Append the last paragraph
    if current_paragraph:
        paragraphs.append(" ".join(current_paragraph))

    return "\n\n".join(paragraphs)


raw_text = """
  Company   Leave Policy

Employees can request annual
leave after six months.

Employees are NOT eligible for
unpaid leave during probation.

Contact HR for further details.
"""

cleaned_text = clean_text(raw_text)
print(cleaned_text)

Company Leave Policy

Employees can request annual leave after six months.

Employees are NOT eligible for unpaid leave during probation.

Contact HR for further details.
